# 11 - Product Quantization (PQ)


---

In the previous notebook, we learned Locality Sensitive Hashing (LSH).

LSH speeds up search by reducing the number of candidate vectors.

But another challenge remains.

Modern vector databases may store millions or even billions of embeddings.

These embeddings consume a large amount of memory.

Researchers developed **Product Quantization (PQ)** to compress vectors while still enabling efficient similarity search.

##  History

As embedding models improved,

their dimensions increased.

Examples:

- MiniLM → 384 dimensions
- BGE → 768 dimensions
- OpenAI → 1536 dimensions

Large vector databases required huge amounts of RAM.

Researchers asked:

"Can we compress vectors while preserving most of their useful information?"

This idea became **Product Quantization (PQ)**.

##  Think Like a Researcher

Imagine storing one million HD photos.

The storage requirement is enormous.

Instead,

you compress the photos into JPEG files.

They lose a little detail,

but they occupy much less space.

PQ does something similar for vectors.

Instead of storing every value exactly,

it stores a compressed representation.

In [1]:
dimension = 768

bytes_per_float = 4

memory = dimension * bytes_per_float

print(f"One vector uses {memory} bytes")

One vector uses 3072 bytes


In [2]:
vectors = 1_000_000

memory_gb = (vectors * 768 * 4) / (1024**3)

print(f"Approximate memory: {memory_gb:.2f} GB")

Approximate memory: 2.86 GB


## Observation

One million vectors already require several gigabytes of memory.

Now imagine

100 million vectors.

Memory becomes one of the biggest challenges in vector databases.

##  Basic Idea

Instead of storing

```
[0.42, -0.18, 0.73, ...]
```

exactly,

PQ stores a short code.

Example

```
[15, 8, 31, 2]
```

The code can later be used to approximate the original vector.

In [3]:
import numpy as np

vector = np.arange(16)

vector

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [5]:
subvectors = np.split(vector,4)

subvectors

#Instead of treating the whole vector as one object,

#PQ divides it into smaller pieces.

[array([0, 1, 2, 3]),
 array([4, 5, 6, 7]),
 array([ 8,  9, 10, 11]),
 array([12, 13, 14, 15])]

## Codebooks

Each subvector is matched to a small set of representative vectors.

These representatives are called **codewords**.

The collection of codewords is called a **codebook**.

Instead of storing the original subvector,

we store only the index of the closest codeword.

In [6]:
codebook = {
    0:"Cluster A",
    1:"Cluster B",
    2:"Cluster C",
    3:"Cluster D"
}

compressed = [2,1,0,3]

print(compressed)

[2, 1, 0, 3]


## Product Quantization Pipeline

```
Original Vector

↓

Split into Subvectors

↓

Find Closest Codeword

↓

Store Code Index

↓

Compressed Vector
```

## Why is PQ Faster?

Compressed vectors require less memory.

Smaller memory footprint means

- more vectors fit into RAM
- better cache efficiency
- faster search

## Advantages

✅ Much lower memory usage

✅ Faster search

✅ Scales to very large datasets

✅ Widely used in production vector search

## Limitations

❌ Compression introduces approximation

❌ Similarity scores are approximate

❌ Requires training codebooks

##  Applications

PQ is used in

- FAISS
- Billion-scale vector search
- Recommendation systems
- Image retrieval
- Large RAG systems

In [ ]:
import faiss

dimension = 128

m = 8

bits = 8

index = faiss.IndexPQ(
    dimension,
    m,
    bits
)

print(index)

# dimension → size of each vector
# m → number of subvectors
# bits → bits used to encode each subvector

<faiss.swigfaiss.IndexPQ; proxy of <Swig Object of type 'faiss::IndexPQ *' at 0x00000251934255B0> >


## Flat vs IVF vs PQ

| Method | Idea | Memory | Speed |
|---------|------|--------|-------|
| Flat | Store original vectors | High | Slow |
| IVF | Search fewer clusters | High | Faster |
| PQ | Compress vectors | Low | Faster |

## Summary

Today I learned

- Why memory becomes a challenge
- What Product Quantization is
- Codebooks
- Codewords
- Subvectors
- Compression
- Advantages and limitations

## Think Like a Researcher

IVF speeds up search by reducing the number of candidate vectors.

PQ reduces memory by compressing vectors.

Researchers then asked:

"What if we combine both ideas?"

- IVF → search fewer vectors
- PQ → store vectors more compactly

Combining them gives us **IVFPQ**, one of the most widely used indexes in FAISS.